# Residual Attribution

Fine-grained attribution of residual stream changes: per-component
contribution tracking, cumulative vs incremental effects, directional
attribution, and interference analysis.

**References:**
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Nanda (2022) "Attribution Patching: Activation Patching At Industrial Scale"

## Why This Matters

Residual stream attribution characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_attribution import (
    per_component_residual_contribution,
    cumulative_residual_buildup,
    directional_attribution,
    component_interference_analysis,
    residual_decomposition_at_unembed,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. Per-component residual contribution
contrib = per_component_residual_contribution(model, tokens)
print(f"Dominant component: {contrib['dominant_component']}")
print(f"\nContribution norms:")
sorted_norms = sorted(contrib['contribution_norms'].items(), key=lambda x: -x[1])
for name, norm in sorted_norms:
    bar = '#' * int(norm * 10 / (sorted_norms[0][1] + 1e-10))
    print(f"  {name:>10}: {norm:.4f} {bar}")

In [ ]:
# 2. Cumulative residual buildup
buildup = cumulative_residual_buildup(model, tokens)
print(f"Residual stream evolution:")
for l in range(cfg.n_layers + 1):
    label = f"post L{l-1}" if l > 0 else "embed"
    bar = '#' * int(buildup['residual_norms'][l] * 5)
    print(f"  {label:>10}: norm={buildup['residual_norms'][l]:.4f} {bar}")
print(f"\nDirection changes (radians):")
for l in range(cfg.n_layers):
    print(f"  L{l}: angle={buildup['direction_changes'][l]:.4f}, growth={buildup['growth_rates'][l]:+.4f}")

In [ ]:
# 3. Directional attribution
# Use unembed direction for token 0
W_U = np.array(model.unembed.W_U)
target_dir = W_U[:, 0]
attr = directional_attribution(model, tokens, target_dir)
print(f"Total attribution along token 0 direction: {attr['total_attribution']:.4f}")
print(f"\nPer-component fractions:")
sorted_fracs = sorted(attr['attribution_fractions'].items(), key=lambda x: -abs(x[1]))
for name, frac in sorted_fracs:
    bar = '#' * int(abs(frac) * 30)
    print(f"  {name:>10}: {frac:+.4f} {bar}")

In [ ]:
# 4. Component interference analysis
inter = component_interference_analysis(model, tokens)
print(f"Net alignment: {inter['net_alignment']:+.4f}")
print(f"Constructive pairs: {len(inter['constructive_pairs'])}")
print(f"Destructive pairs: {len(inter['destructive_pairs'])}")
print(f"\nTop constructive:")
sorted_align = sorted(inter['pairwise_alignment'].items(), key=lambda x: -x[1])
for pair, cos in sorted_align[:5]:
    print(f"  {pair[0]:>10} + {pair[1]:<10}: {cos:+.4f}")
print(f"Top destructive:")
for pair, cos in sorted_align[-5:]:
    print(f"  {pair[0]:>10} + {pair[1]:<10}: {cos:+.4f}")

In [ ]:
# 5. Residual decomposition at unembed
decomp = residual_decomposition_at_unembed(model, tokens)
print(f"Top tokens per component (token_idx: logit_value):")
for comp, top_list in decomp['top_tokens_per_component'].items():
    tokens_str = ', '.join(f"t{t}({v:+.3f})" for t, v in top_list[:3])
    print(f"  {comp:>10}: {tokens_str}")
print(f"\nTotal logits (top 5 tokens):")
top_idx = np.argsort(decomp['total_logits'])[::-1][:5]
for i in top_idx:
    print(f"  Token {i}: {decomp['total_logits'][i]:+.4f}")